# Taiwan Credit Card Default — Exploratory Data Analysis

## Dataset: Default of Credit Card Clients

**Source:** [UCI Machine Learning Repository – Dataset #350](https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients)

This dataset contains information on **default payments, demographic factors, credit limit, payment history, and bill statements** of credit card clients in **Taiwan** from **April 2005 to September 2005**.

| Attribute | Detail |
|-----------|--------|
| Instances | 30,000 credit card clients |
| Features | 23 input features + 1 target variable |
| Domain | Finance / Credit Risk Modelling |
| Target | `Default_Payment_Next_Month` — 1 = Default, 0 = No Default |
| Missing Values | None |

### EDA Flow
1. **Data Import** — Load dataset from UCI repository  
2. **Data Cleaning** — Fix column names and undocumented category codes  
3. **Data Overview** — Shape, types, descriptive statistics  
4. **EDA** — Univariate & bivariate analysis, correlation, outliers  
5. **Conclusion** — Key insights for modelling  
6. **HTML Export** — Generate static report

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import json
import warnings

warnings.filterwarnings('ignore')

# Plot style
plt.rcParams['figure.dpi'] = 100
plt.rcParams['font.size']  = 11
sns.set_style('whitegrid')
sns.set_palette('viridis')

print("Libraries imported successfully.")

In [3]:
from ucimlrepo import fetch_ucirepo
default_of_credit_card_clients = fetch_ucirepo(id=350) 
  
# data (as pandas dataframes) 
X = default_of_credit_card_clients.data.features 
y = default_of_credit_card_clients.data.targets 
  
# metadata 
print(default_of_credit_card_clients.metadata) 
  
# variable information 
print(default_of_credit_card_clients.variables) 


{'uci_id': 350, 'name': 'Default of Credit Card Clients', 'repository_url': 'https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients', 'data_url': 'https://archive.ics.uci.edu/static/public/350/data.csv', 'abstract': "This research aimed at the case of customers' default payments in Taiwan and compares the predictive accuracy of probability of default among six data mining methods.", 'area': 'Business', 'tasks': ['Classification'], 'characteristics': ['Multivariate'], 'num_instances': 30000, 'num_features': 23, 'feature_types': ['Integer', 'Real'], 'demographics': ['Sex', 'Education Level', 'Marital Status', 'Age'], 'target_col': ['Y'], 'index_col': ['ID'], 'has_missing_values': 'no', 'missing_values_symbol': None, 'year_of_dataset_creation': 2009, 'last_updated': 'Fri Mar 29 2024', 'dataset_doi': '10.24432/C55S3H', 'creators': ['I-Cheng Yeh'], 'intro_paper': {'ID': 365, 'type': 'NATIVE', 'title': 'The comparisons of data mining techniques for the predictive accuracy of 

## 1. Data Cleaning

The raw dataset uses numeric codes for column names (`X1`–`X23`, `Y`) and contains undocumented category values in some fields. The following steps are applied:

| Step | Action |
|------|--------|
| 1 | **Rename columns** — `X1`–`X23` → descriptive names; `Y` → `Default_Payment_Next_Month` |
| 2 | **Fix `Education`** — Undocumented codes (0, 5, 6) mapped to 4 (Others) |
| 3 | **Fix `Marriage`** — Undocumented code (0) mapped to 3 (Others) |

**Payment Status coding (PAY_1–PAY_6):**  
`-2` = No consumption · `-1` = Paid in full · `0` = Revolving credit  
`1`–`8` = Payment delay by 1–8 months

In [ ]:
import pandas as pd
import numpy as np

# 1. Combine features and target into a single DataFrame
df = pd.concat([X, y], axis=1)

# 2. Rename columns from X1–X23 to descriptive names
column_mapping = {
    'X1': 'Credit_Limit', 'X2': 'Gender', 'X3': 'Education', 'X4': 'Marriage', 'X5': 'Age',
    'X6': 'PAY_1',  'X7': 'PAY_2',  'X8': 'PAY_3',
    'X9': 'PAY_4',  'X10': 'PAY_5', 'X11': 'PAY_6',
    'X12': 'BILL_AMT_1', 'X13': 'BILL_AMT_2', 'X14': 'BILL_AMT_3',
    'X15': 'BILL_AMT_4', 'X16': 'BILL_AMT_5', 'X17': 'BILL_AMT_6',
    'X18': 'PAY_AMT_1', 'X19': 'PAY_AMT_2', 'X20': 'PAY_AMT_3',
    'X21': 'PAY_AMT_4', 'X22': 'PAY_AMT_5', 'X23': 'PAY_AMT_6',
    'Y': 'Default_Payment_Next_Month'
}
df.rename(columns=column_mapping, inplace=True)

# 3. Fix Education: undocumented categories (0, 5, 6) → 4 (Others)
df['Education'] = df['Education'].replace([0, 5, 6], 4)

# 4. Fix Marriage: undocumented category (0) → 3 (Others)
df['Marriage'] = df['Marriage'].replace(0, 3)

# 5. Verify cleaning results
print("=== Data Cleaning Complete ===")
print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Missing values: {df.isnull().sum().sum()}")
print(f"Education unique values: {sorted(df['Education'].unique())}")
print(f"Marriage  unique values: {sorted(df['Marriage'].unique())}")
print(f"PAY_1     unique values: {sorted(df['PAY_1'].unique())}")
df.head(3)

## 2. Data Overview

Examine the shape, data types, descriptive statistics, and missing values before any visualisation.

In [ ]:
import pandas as pd

print("=" * 60)
print("DATASET SHAPE")
print("=" * 60)
print(f"Rows: {df.shape[0]:,}   |   Columns: {df.shape[1]}")

print("\n" + "=" * 60)
print("DATA TYPES & MISSING VALUES")
print("=" * 60)
overview_df = pd.DataFrame({
    'dtype':     df.dtypes,
    'non_null':  df.notnull().sum(),
    'missing':   df.isnull().sum(),
    'missing_%': (df.isnull().sum() / len(df) * 100).round(2),
    'unique':    df.nunique()
})
display(overview_df)

print("\n" + "=" * 60)
print("STATISTICAL SUMMARY — Numerical Features")
print("=" * 60)
num_cols_overview = [
    'Credit_Limit', 'Age',
    'BILL_AMT_1', 'BILL_AMT_2', 'BILL_AMT_3',
    'PAY_AMT_1',  'PAY_AMT_2',  'PAY_AMT_3'
]
display(df[num_cols_overview].describe().round(2))

print("\n" + "=" * 60)
print("CATEGORICAL FEATURE VALUE COUNTS")
print("=" * 60)
cat_map = {
    'Gender':    {1: 'Male', 2: 'Female'},
    'Education': {1: 'Graduate School', 2: 'University', 3: 'High School', 4: 'Others'},
    'Marriage':  {1: 'Married', 2: 'Single', 3: 'Others'},
    'Default_Payment_Next_Month': {0: 'No Default', 1: 'Default'}
}
for col, labels in cat_map.items():
    counts = df[col].value_counts().sort_index()
    print(f"\n{col}:")
    for k, v in counts.items():
        print(f"  {labels.get(k, str(k))} ({k}): {v:,}  ({v / len(df) * 100:.1f}%)")

# Store overview data for HTML
overview_data = {
    'n_rows':    df.shape[0],
    'n_cols':    df.shape[1],
    'n_missing': int(df.isnull().sum().sum()),
    'dtypes':    df.dtypes.value_counts().to_dict(),
    'cat_distributions': {col: {labels.get(k, str(k)): int(v) for k, v in df[col].value_counts().sort_index().items()}
                          for col, labels in cat_map.items()}
}

## 3. Exploratory Data Analysis (EDA)

The analyses performed in this section:

| Section | Method | Purpose |
|---------|--------|---------|
| 3.1 | **Target Distribution** | Check class balance — informs sampling strategy |
| 3.2 | **Variable Analysis** (Univariate + Bivariate) | Distribution of each feature; relationship with default target |
| 3.3 | **Correlation Matrix** (Pearson) | Identify linearly correlated features and potential multicollinearity |
| 3.4 | **Outlier Detection** (IQR method) | Flag extreme values in numerical features |
| 3.5 | **Sample Rows by Class** | Inspect real records from each target class |

### 3.1 Target Variable Distribution

The target variable `Default_Payment_Next_Month` indicates whether a client defaulted (`1`) or not (`0`).  
A class imbalance check is essential — it affects classifier choice and which evaluation metrics (e.g. F1, AUC-ROC) should be used.

In [ ]:
import matplotlib.pyplot as plt
import json

target_counts = df['Default_Payment_Next_Month'].value_counts().sort_index()
labels  = ['No Default (0)', 'Default (1)']
sizes   = [int(target_counts[0]), int(target_counts[1])]
colors  = ['#4CAF50', '#F44336']
explode = (0, 0.08)

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
fig.suptitle('Target Variable Distribution — Default Payment Next Month',
             fontsize=14, fontweight='bold')

# Pie chart
axes[0].pie(sizes, labels=labels, colors=colors, explode=explode,
            autopct='%1.1f%%', startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2},
            textprops={'fontsize': 12})
axes[0].set_title('Class Distribution (Pie)', fontsize=12)

# Bar chart
bars = axes[1].bar(labels, sizes, color=colors, edgecolor='white', linewidth=1.5)
axes[1].set_title('Class Counts (Bar)', fontsize=12)
axes[1].set_ylabel('Count')
for bar, count in zip(bars, sizes):
    axes[1].text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 200,
                 f'{count:,}\n({count / sum(sizes) * 100:.1f}%)',
                 ha='center', fontsize=11, fontweight='bold')
axes[1].set_ylim(0, max(sizes) * 1.25)

plt.tight_layout()
plt.show()

imbalance = sizes[0] / sizes[1]
print(f"No Default: {sizes[0]:,} ({sizes[0] / sum(sizes) * 100:.1f}%)")
print(f"Default:    {sizes[1]:,} ({sizes[1] / sum(sizes) * 100:.1f}%)")
print(f"Imbalance ratio (No Default / Default): {imbalance:.2f}:1")

# Export
target_data = {
    'labels':      labels,
    'counts':      sizes,
    'percentages': [round(s / sum(sizes) * 100, 2) for s in sizes]
}

### 3.2 Variable Analysis

For each feature group:
- **Univariate**: distribution / frequency of values  
- **Bivariate**: mean default rate per category or bin (target relationship)

#### 3.2.1 Payment Status Variables (PAY_1 – PAY_6)

`PAY_1`–`PAY_6` represent the repayment status for **September → April 2005** respectively:

| Code | Meaning |
|------|---------|
| -2 | No consumption |
| -1 | Paid in full (duly) |
|  0 | Revolving credit used |
|  1 | 1-month payment delay |
|  2 | 2-month payment delay |
| ... | ... |
|  8 | 8+ month payment delay |

Higher delay codes are expected to strongly correlate with default.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import json

pay_cols = ['PAY_1', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']

# ── Plot 1: Distribution ─────────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Payment Status Distribution (PAY_1 ~ PAY_6)', fontsize=15, fontweight='bold')

for ax, col in zip(axes.flat, pay_cols):
    val_counts = df[col].value_counts().sort_index()
    ax.bar(val_counts.index.astype(str), val_counts.values,
           color='steelblue', edgecolor='white')
    ax.set_title(col, fontsize=12)
    ax.set_xlabel('Status Code')
    ax.set_ylabel('Count')

plt.tight_layout()
plt.show()

# ── Plot 2: Default Rate by Payment Status ───────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Default Rate by Payment Status (PAY_1 ~ PAY_6)', fontsize=15, fontweight='bold')

for ax, col in zip(axes.flat, pay_cols):
    rate = df.groupby(col)['Default_Payment_Next_Month'].mean().reset_index()
    ax.bar(rate[col].astype(str), rate['Default_Payment_Next_Month'],
           color='salmon', edgecolor='white')
    ax.set_title(col, fontsize=12)
    ax.set_xlabel('Status Code')
    ax.set_ylabel('Default Rate')
    ax.set_ylim(0, 1)
    for i, (x, y_val) in enumerate(zip(rate[col].astype(str), rate['Default_Payment_Next_Month'])):
        ax.text(i, y_val + 0.02, f'{y_val:.2f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

# ── Export JSON for HTML ─────────────────────────────────────────────────────
pay_status_data = {}
for col in pay_cols:
    val_counts = df[col].value_counts().sort_index()
    default_rate = df.groupby(col)['Default_Payment_Next_Month'].mean()
    pay_status_data[col] = {
        'categories': [int(k) for k in val_counts.index],
        'counts':     [int(v) for v in val_counts.values],
        'default_rates': [round(float(default_rate.get(k, 0)), 4) for k in val_counts.index]
    }

print("Pay status data ready for HTML export.")

#### 3.2.2 Demographic Variables (Gender, Education, Marriage)

Categorical demographics are examined for both their individual distribution and their relationship with the default rate.

| Feature | Categories |
|---------|-----------|
| Gender | 1 = Male, 2 = Female |
| Education | 1 = Graduate School, 2 = University, 3 = High School, 4 = Others |
| Marriage | 1 = Married, 2 = Single, 3 = Others |

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

cat_features = {
    'Gender':    {1: 'Male', 2: 'Female'},
    'Education': {1: 'Grad School', 2: 'University', 3: 'High School', 4: 'Others'},
    'Marriage':  {1: 'Married', 2: 'Single', 3: 'Others'}
}

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.suptitle('Categorical Variables — Distribution & Default Rate', fontsize=15, fontweight='bold')

for i, (col, labels) in enumerate(cat_features.items()):
    val_counts  = df[col].value_counts().sort_index()
    x_labels    = [labels.get(k, str(k)) for k in val_counts.index]
    default_rate = df.groupby(col)['Default_Payment_Next_Month'].mean()
    x_rates     = [labels.get(k, str(k)) for k in default_rate.index]

    # Distribution (top row)
    ax_top = axes[0, i]
    bars = ax_top.bar(x_labels, val_counts.values,
                      color=sns.color_palette('Set2', len(val_counts)), edgecolor='white')
    ax_top.set_title(f'{col} — Distribution', fontsize=12)
    ax_top.set_ylabel('Count')
    for bar, v in zip(bars, val_counts.values):
        ax_top.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 50,
                    f'{v / len(df) * 100:.1f}%', ha='center', fontsize=9)

    # Default rate (bottom row)
    ax_bot = axes[1, i]
    bars2 = ax_bot.bar(x_rates, default_rate.values, color='salmon', edgecolor='white')
    ax_bot.set_title(f'{col} — Default Rate', fontsize=12)
    ax_bot.set_ylabel('Default Rate')
    ax_bot.set_ylim(0, 0.6)
    for bar, v in zip(bars2, default_rate.values):
        ax_bot.text(bar.get_x() + bar.get_width() / 2, v + 0.01,
                    f'{v:.3f}', ha='center', fontsize=9)

plt.tight_layout()
plt.show()

# ── Export JSON ───────────────────────────────────────────────────────────────
categorical_data = {}
for col, labels in cat_features.items():
    val_counts   = df[col].value_counts().sort_index()
    default_rate = df.groupby(col)['Default_Payment_Next_Month'].mean()
    categorical_data[col] = {
        'categories':    [labels.get(k, str(k)) for k in val_counts.index],
        'counts':        [int(v) for v in val_counts.values],
        'default_rates': [round(float(default_rate.get(k, 0)), 4) for k in val_counts.index]
    }

print("Categorical data ready for HTML export.")

#### 3.2.3 Numerical Variables — Credit Limit & Age

- **`Credit_Limit`**: Amount of given credit (NT dollars). Clients with lower limits may have higher default risk.  
- **`Age`**: Client age in years. Binned to explore age group differences in default behaviour.

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle('Credit Limit & Age — Distribution and Default Rate', fontsize=14, fontweight='bold')

configs = [
    ('Credit_Limit',
     [0, 50000, 100000, 200000, 300000, 500000, 800000, 1000000],
     ['0–50K', '50–100K', '100–200K', '200–300K', '300–500K', '500–800K', '800K+']),
    ('Age',
     [20, 25, 30, 35, 40, 50, 60, 80],
     ['20–25', '25–30', '30–35', '35–40', '40–50', '50–60', '60+'])
]

numerical_data = {}
for i, (col, bins, bin_labels) in enumerate(configs):
    # Distribution histogram
    ax_hist = axes[0, i]
    ax_hist.hist(df[col], bins=40, color='steelblue', edgecolor='white', alpha=0.85)
    ax_hist.axvline(df[col].mean(),   color='red',    linestyle='--',
                    label=f'Mean: {df[col].mean():.1f}')
    ax_hist.axvline(df[col].median(), color='orange', linestyle='--',
                    label=f'Median: {df[col].median():.1f}')
    ax_hist.set_title(f'{col} — Distribution', fontsize=12)
    ax_hist.set_xlabel(col)
    ax_hist.set_ylabel('Count')
    ax_hist.legend(fontsize=9)

    # Default rate by bin
    ax_rate = axes[1, i]
    tmp = df.copy()
    tmp['bin'] = pd.cut(tmp[col], bins=bins, labels=bin_labels, right=False)
    rate   = tmp.groupby('bin', observed=True)['Default_Payment_Next_Month'].mean()
    counts = tmp.groupby('bin', observed=True)['Default_Payment_Next_Month'].count()
    bars = ax_rate.bar(rate.index, rate.values, color='coral', edgecolor='white')
    ax_rate.set_title(f'{col} — Default Rate by Bin', fontsize=12)
    ax_rate.set_xlabel('Bin')
    ax_rate.set_ylabel('Default Rate')
    ax_rate.set_ylim(0, 0.65)
    for bar, v, c in zip(bars, rate.values, counts.values):
        ax_rate.text(bar.get_x() + bar.get_width() / 2, v + 0.01,
                     f'{v:.2f}\n(n={c:,})', ha='center', fontsize=8)
    plt.setp(ax_rate.get_xticklabels(), rotation=30, ha='right')

    numerical_data[col] = {
        'bins':          bin_labels,
        'counts':        [int(c) for c in counts.values],
        'default_rates': [round(float(r), 4) for r in rate.values],
        'mean':   round(float(df[col].mean()), 2),
        'median': round(float(df[col].median()), 2),
        'std':    round(float(df[col].std()), 2)
    }

plt.tight_layout()
plt.show()
print("Numerical feature data ready for HTML export.")
print(f"Credit_Limit — mean={df['Credit_Limit'].mean():,.0f}, median={df['Credit_Limit'].median():,.0f}")
print(f"Age          — mean={df['Age'].mean():.1f}, range=[{df['Age'].min()}, {df['Age'].max()}]")

#### 3.2.4 Bill Amount Variables (BILL_AMT_1 – BILL_AMT_6)

Bill statement amounts for the 6 preceding months (September → April 2005).  
Negative values indicate a **credit balance** (client overpaid). Highly right-skewed distribution is expected.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

bill_cols = ['BILL_AMT_1', 'BILL_AMT_2', 'BILL_AMT_3', 'BILL_AMT_4', 'BILL_AMT_5', 'BILL_AMT_6']

# Distributions
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Bill Amount Distributions (September → April 2005)', fontsize=14, fontweight='bold')
for ax, col in zip(axes.flat, bill_cols):
    ax.hist(df[col], bins=50, color='mediumpurple', edgecolor='white', alpha=0.85)
    ax.axvline(df[col].mean(),   color='red',    linestyle='--', linewidth=1.5,
               label=f'Mean: {df[col].mean() / 1000:.0f}K')
    ax.axvline(df[col].median(), color='orange', linestyle='--', linewidth=1.5,
               label=f'Median: {df[col].median() / 1000:.0f}K')
    ax.set_title(col, fontsize=12)
    ax.set_xlabel('Amount (NT$)')
    ax.set_ylabel('Count')
    ax.legend(fontsize=8)
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x / 1000:.0f}K'))
plt.tight_layout()
plt.show()

# Default rate by quintile
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Default Rate by Bill Amount (Quintile Bins)', fontsize=14, fontweight='bold')
bill_data = {}
for ax, col in zip(axes.flat, bill_cols):
    tmp = df[[col, 'Default_Payment_Next_Month']].copy()
    tmp['qbin'] = pd.qcut(tmp[col], q=5, duplicates='drop')
    rate   = tmp.groupby('qbin', observed=True)['Default_Payment_Next_Month'].mean()
    counts = tmp.groupby('qbin', observed=True)['Default_Payment_Next_Month'].count()
    qlabels = [f'Q{i+1}' for i in range(len(rate))]
    ax.bar(qlabels, rate.values, color='skyblue', edgecolor='white')
    ax.set_title(col, fontsize=12)
    ax.set_xlabel('Quintile')
    ax.set_ylabel('Default Rate')
    ax.set_ylim(0, 0.55)
    for i, (v, c) in enumerate(zip(rate.values, counts.values)):
        ax.text(i, v + 0.01, f'{v:.2f}', ha='center', fontsize=9)
    bill_data[col] = {
        'quintile_counts':  [int(c) for c in counts.values],
        'default_rates':    [round(float(r), 4) for r in rate.values],
        'mean':   round(float(df[col].mean()), 2),
        'median': round(float(df[col].median()), 2),
        'std':    round(float(df[col].std()), 2)
    }
plt.tight_layout()
plt.show()
print("Bill amount data ready for HTML export.")

#### 3.2.5 Payment Amount Variables (PAY_AMT_1 – PAY_AMT_6)

Amount actually paid each month (previous 6 months, September → April 2005).  
Zero values mean **no payment was made** — expected to correlate strongly with defaults.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

pay_amt_cols = ['PAY_AMT_1', 'PAY_AMT_2', 'PAY_AMT_3', 'PAY_AMT_4', 'PAY_AMT_5', 'PAY_AMT_6']

# Distribution (log scale for non-zero payments)
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Payment Amount Distributions (log₁₀ of non-zero values)', fontsize=14, fontweight='bold')
for ax, col in zip(axes.flat, pay_amt_cols):
    non_zero  = df[df[col] > 0][col]
    zero_pct  = (df[col] == 0).sum() / len(df) * 100
    ax.hist(np.log10(non_zero + 1), bins=40, color='mediumseagreen', edgecolor='white', alpha=0.85)
    ax.set_title(f'{col}\n(0 payments: {zero_pct:.1f}%)', fontsize=11)
    ax.set_xlabel('log₁₀(Amount + 1)')
    ax.set_ylabel('Count')
plt.tight_layout()
plt.show()

# Default rate: zero vs non-zero payment
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle('Default Rate — Zero vs Non-Zero Payment Amount', fontsize=14, fontweight='bold')
pay_amt_data = {}
for ax, col in zip(axes.flat, pay_amt_cols):
    grp = df.copy()
    grp['payment_group'] = grp[col].apply(lambda x: 'No Payment (0)' if x == 0 else 'Paid (>0)')
    rate   = grp.groupby('payment_group')['Default_Payment_Next_Month'].mean()
    counts = grp.groupby('payment_group')['Default_Payment_Next_Month'].count()
    bars = ax.bar(rate.index, rate.values,
                  color=['salmon', 'mediumseagreen'], edgecolor='white')
    ax.set_title(col, fontsize=12)
    ax.set_ylabel('Default Rate')
    ax.set_ylim(0, 0.65)
    for bar, v, c in zip(bars, rate.values, counts.values):
        ax.text(bar.get_x() + bar.get_width() / 2, v + 0.01,
                f'{v:.2f}\n(n={c:,})', ha='center', fontsize=9)
    pay_amt_data[col] = {
        'mean':               round(float(df[col].mean()), 2),
        'median':             round(float(df[col].median()), 2),
        'zero_pct':           round(float((df[col] == 0).sum() / len(df) * 100), 2),
        'default_rate_zero':  round(float(rate.get('No Payment (0)', 0)), 4),
        'default_rate_paid':  round(float(rate.get('Paid (>0)', 0)), 4)
    }
plt.tight_layout()
plt.show()
print("Payment amount data ready for HTML export.")

### 3.3 Correlation Matrix

**Method:** Pearson correlation coefficients between all numerical features and the target variable.

**Why?** It is the standard, most interpretable way to measure pairwise linear association. Values close to **+1** or **−1** indicate strong linear relationships; values near **0** indicate weak/no linear correlation. This also reveals potential **multicollinearity** between predictor features.

$$r_{xy} = \frac{\sum_{i=1}^{n}(x_i - \bar{x})(y_i - \bar{y})}{\sqrt{\sum_{i=1}^{n}(x_i - \bar{x})^2 \cdot \sum_{i=1}^{n}(y_i - \bar{y})^2}}$$

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import json

# Numerical columns for correlation (exclude categorical-coded columns)
numerical_cols = [
    'Credit_Limit', 'Age',
    'BILL_AMT_1', 'BILL_AMT_2', 'BILL_AMT_3', 'BILL_AMT_4', 'BILL_AMT_5', 'BILL_AMT_6',
    'PAY_AMT_1',  'PAY_AMT_2',  'PAY_AMT_3',  'PAY_AMT_4',  'PAY_AMT_5',  'PAY_AMT_6',
    'Default_Payment_Next_Month'
]
numerical_cols = [c for c in numerical_cols if c in df.columns]

correlation_matrix = df[numerical_cols].corr()

plt.figure(figsize=(14, 12))
mask = None  # show all
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f',
            linewidths=0.5, square=True, cbar_kws={'shrink': 0.8})
plt.title('Correlation Matrix — Numerical Features', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# Top correlations with target
target_corr = (correlation_matrix['Default_Payment_Next_Month']
               .drop('Default_Payment_Next_Month')
               .abs()
               .sort_values(ascending=False))
print("Top 10 features correlated with Default_Payment_Next_Month:")
print(target_corr.head(10).round(4))

# Export for HTML
correlation_data = {
    'columns': list(correlation_matrix.columns),
    'matrix':  [[round(float(v), 4) for v in row]
                for row in correlation_matrix.values]
}
print("\nCorrelation data ready for HTML export.")

### 3.4 Outlier Detection — IQR Method

The **Interquartile Range (IQR)** method is used to identify outliers in numerical features:

$$\text{Lower bound} = Q_1 - 1.5 \times IQR \qquad \text{Upper bound} = Q_3 + 1.5 \times IQR$$

Any observation outside this range is flagged as an outlier.

**Why IQR?**  
- Robust to skewed distributions (unlike z-score which assumes normality)  
- Widely accepted for financial data with heavy tails  
- Does not require assumptions about the underlying distribution

Severity classification:  `low` < 3% · `medium` 3–10% · `high` > 10%

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

outlier_cols = [
    'Credit_Limit', 'Age',
    'BILL_AMT_1', 'BILL_AMT_2', 'BILL_AMT_3', 'BILL_AMT_4', 'BILL_AMT_5', 'BILL_AMT_6',
    'PAY_AMT_1',  'PAY_AMT_2',  'PAY_AMT_3',  'PAY_AMT_4',  'PAY_AMT_5',  'PAY_AMT_6'
]

# ── Box plots ────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(3, 5, figsize=(22, 12))
fig.suptitle('Box Plots — Outlier Detection (IQR Method)', fontsize=14, fontweight='bold')
for ax, col in zip(axes.flat, outlier_cols):
    ax.boxplot(df[col].dropna(), vert=True, patch_artist=True,
               boxprops=dict(facecolor='lightblue', color='steelblue'),
               medianprops=dict(color='red', linewidth=2))
    ax.set_title(col, fontsize=9)
    ax.set_xticks([])
# Hide unused axes
for ax in axes.flat[len(outlier_cols):]:
    ax.set_visible(False)
plt.tight_layout()
plt.show()

# ── IQR statistics ───────────────────────────────────────────────────────────
outlier_stats = {}
for col in outlier_cols:
    if col in df.columns:
        Q1 = df[col].quantile(0.25)
        Q3 = df[col].quantile(0.75)
        IQR = Q3 - Q1
        lower = Q1 - 1.5 * IQR
        upper = Q3 + 1.5 * IQR
        n_out = int(((df[col] < lower) | (df[col] > upper)).sum())
        pct   = round(n_out / len(df) * 100, 2)
        severity = 'high' if pct > 10 else ('medium' if pct > 3 else 'low')
        outlier_stats[col] = {
            'num_outliers':         n_out,
            'percentage_outliers':  pct,
            'severity':             severity,
            'lower_bound':          round(float(lower), 2),
            'upper_bound':          round(float(upper), 2),
            'Q1': round(float(Q1), 2),
            'Q3': round(float(Q3), 2),
            'IQR': round(float(IQR), 2)
        }

print(f"{'Column':<14} {'Outliers':>8} {'%':>7} {'Severity':<10} {'Lower':>12} {'Upper':>12}")
print("-" * 65)
for col, s in outlier_stats.items():
    print(f"{col:<14} {s['num_outliers']:>8,} {s['percentage_outliers']:>6.1f}%  "
          f"{s['severity']:<10} {s['lower_bound']:>12,.1f} {s['upper_bound']:>12,.1f}")

print("\nOutlier data ready for HTML export.")

### 3.5 Sample Rows by Target Class

Inspect a random sample of records from each class to provide intuitive evidence of patterns distinguishing defaulters from non-defaulters.

In [ ]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:,.0f}'.format)

KEY_COLS = ['Credit_Limit', 'Gender', 'Education', 'Marriage', 'Age',
            'PAY_1', 'PAY_2', 'BILL_AMT_1', 'PAY_AMT_1', 'Default_Payment_Next_Month']

print("=" * 70)
print("SAMPLE ROWS — No Default (target = 0)")
print("=" * 70)
sample_no_default = df[df['Default_Payment_Next_Month'] == 0].sample(5, random_state=42)
display(sample_no_default[KEY_COLS])

print("\n" + "=" * 70)
print("SAMPLE ROWS — Default (target = 1)")
print("=" * 70)
sample_default = df[df['Default_Payment_Next_Month'] == 1].sample(5, random_state=42)
display(sample_default[KEY_COLS])